---

# Day 4: Repose Record

---

ou've sneaked into another supply closet - this time, it's across from the prototype suit manufacturing lab. You need to sneak inside and fix the issues with the suit, but there's a guard stationed outside the lab, so this is as close as you can safely get.

As you search the closet for anything that might help, you discover that you're not the first person to want to sneak in. Covering the walls, someone has spent an hour starting every midnight for the past few months secretly observing this guard post! They've been writing down the ID of the one guard on duty that night - the Elves seem to have decided that one guard was enough for the overnight shift - as well as when they fall asleep or wake up while at their post (your puzzle input).

For example, consider the following records, which have already been organized into chronological order:

    [1518-11-01 00:00] Guard #10 begins shift
    [1518-11-01 00:05] falls asleep
    [1518-11-01 00:25] wakes up
    [1518-11-01 00:30] falls asleep
    [1518-11-01 00:55] wakes up
    [1518-11-01 23:58] Guard #99 begins shift
    [1518-11-02 00:40] falls asleep
    [1518-11-02 00:50] wakes up
    [1518-11-03 00:05] Guard #10 begins shift
    [1518-11-03 00:24] falls asleep
    [1518-11-03 00:29] wakes up
    [1518-11-04 00:02] Guard #99 begins shift
    [1518-11-04 00:36] falls asleep
    [1518-11-04 00:46] wakes up
    [1518-11-05 00:03] Guard #99 begins shift
    [1518-11-05 00:45] falls asleep
    [1518-11-05 00:55] wakes up

Timestamps are written using year-month-day hour:minute format. The guard falling asleep or waking up is always the one whose shift most recently started. Because all asleep/awake times are during the midnight hour (00:00 - 00:59), only the minute portion (00 - 59) is relevant for those events.

Visually, these records show that the guards are asleep at these times:

    Date   ID   Minute
                000000000011111111112222222222333333333344444444445555555555
                012345678901234567890123456789012345678901234567890123456789
    11-01  #10  .....####################.....#########################.....
    11-02  #99  ........................................##########..........
    11-03  #10  ........................#####...............................
    11-04  #99  ....................................##########..............
    11-05  #99  .............................................##########.....

The columns are Date, which shows the month-day portion of the relevant day; ID, which shows the guard on duty that day; and Minute, which shows the minutes during which the guard was asleep within the midnight hour. (The Minute column's header shows the minute's ten's digit in the first row and the one's digit in the second row.) Awake is shown as ., and asleep is shown as #.

Note that guards count as asleep on the minute they fall asleep, and they count as awake on the minute they wake up. For example, because Guard #10 wakes up at 00:25 on 1518-11-01, minute 25 is marked as awake.

If you can figure out the guard most likely to be asleep at a specific time, you might be able to trick that guard into working tonight so you can have the best chance of sneaking in. You have two strategies for choosing the best guard/minute combination.

Strategy 1: Find the guard that has the most minutes asleep. What minute does that guard spend asleep the most?

In the example above, Guard #10 spent the most minutes asleep, a total of 50 minutes (20+25+5), while Guard #99 only slept for a total of 30 minutes (10+10+10). Guard #10 was asleep most during minute 24 (on two days, whereas any other minute the guard was asleep was only seen on one day).

While this example listed the entries in chronological order, your entries are in the order you found them. You'll need to organize them before they can be analyzed.

What is the ID of the guard you chose multiplied by the minute you chose? (In the above example, the answer would be 10 * 24 = 240.)

---

## Utilisation des données d'entrainements

In [22]:
from numpy.distutils.fcompiler import none

fichier = open("input_test.txt", "r")
lignes = fichier.readlines()

verbose = True

---

## Utilisation des données du challenges

In [27]:
fichier = open("input_challenge.txt", "r")
lignes = fichier.readlines()

verbose = False

---
## Etape 1

In [28]:
from collections import Counter
from datetime import datetime
from datetime import timedelta

import re

# Création d'une variable qui va stocker le résultat du challenge
resultatChallenge = 0


# Fonction pour extraire la date et la convertir en objet datetime
def extraire_date(log):
    date_str = log[1:17]  # Extraire "YYYY-MM-DD HH:MM"
    return datetime.strptime(date_str, "%Y-%m-%d %H:%M")

# Trier les logs par date
logs_ordonnees = sorted(lignes, key=extraire_date)

# Variable qui va symboliser l'id du guard travaillant ce soir la
currentGuard = None
startSleep = None

dicoGuarde = {}
dicoPlage = {}

for log in logs_ordonnees:
    event = log[19:].strip()
    if verbose: print(f"{log=}")

    if event.startswith("Guard"):
        currentGuard = re.findall("\d+",event)[0]
        startSleep = None

        if verbose: print(f" >> {currentGuard=}")

    elif event.startswith("falls"):
        startSleep = extraire_date(log)
        if verbose: print(f" >> StartSleep")

    elif event.startswith("wakes") and startSleep != None:
        endSleep = extraire_date(log)
        # On peut calculer le nombre de seconde en utilisant une soustraction puis en divisant par 60
        diff = (endSleep - startSleep) / 60
        tuple = (startSleep.minute,endSleep.minute)
        guard = dicoGuarde.get(currentGuard)

        if guard:
            dicoGuarde[currentGuard] = guard + diff
            dicoPlage[currentGuard].append(tuple)

        else :
            dicoGuarde[currentGuard] = diff
            dicoPlage[currentGuard] = [tuple]


        if verbose: print(f" >> endsleep durée : {diff=}")

##################
# Seconde étape
# On recherche le guarde qui à le plus dormi

maxSleepingGuard = 0
maxSleep = timedelta()
for guard,sleepingTime in dicoGuarde.items():
      if sleepingTime > maxSleep:
          maxSleepingGuard = guard
          maxSleep = sleepingTime

#################
# Troisième étape
# On ajoute chaque minute des plages dans une liste
# et on regarde la minute la plus présente

minutesSleeping = []

for start,end in dicoPlage[maxSleepingGuard]:
    for i in range(start,end):
        minutesSleeping.append(i)

compteur = Counter(minutesSleeping)
max_occurrences = max(compteur.values())

# Filtrer les numéros avec la fréquence maximale
numeros_le_plus_frequents = [numero for numero, nb in compteur.items() if nb == max_occurrences]

print(f"Guarde : {maxSleepingGuard}, Les minutes {numeros_le_plus_frequents} apparaissent {max_occurrences} fois.")

resultatChallenge = int(maxSleepingGuard) * numeros_le_plus_frequents[0]

print(f"\nRésultat du challenge partie 2 : {resultatChallenge}")

Guarde : 1571, Les minutes [54] apparaissent 12 fois.

Résultat du challenge partie 2 : 84834


---
## Etape 2

In [30]:
# Création d'une variable qui va stocker le résultat du challenge
resultatChallenge = 0

# On va reprendre les données du tabeau précedent

listEtape2 = []

for guard,plage in dicoPlage.items():
    minutesSleeping = []
    for start,end in plage:
        for i in range(start,end):
            minutesSleeping.append(i)

    if verbose: print(f" >> {guard=}, {minutesSleeping=}")
    compteur = Counter(minutesSleeping)
    max_occurrences = max(compteur.values())

    # Filtrer les numéros avec la fréquence maximale
    numeros_le_plus_frequents = [numero for numero, nb in compteur.items() if nb == max_occurrences]

    listEtape2.append((guard,numeros_le_plus_frequents,max_occurrences))

for a,b,c in listEtape2:
    print(f"Guarde : {a}, Les minutes {b} apparaissent {c} fois.")


print(f"\nRésultat du challenge partie 2 : {33*1619}")

Guarde : 2543, Les minutes [39] apparaissent 6 fois.
Guarde : 3571, Les minutes [38] apparaissent 9 fois.
Guarde : 1697, Les minutes [34, 35, 36, 37, 38] apparaissent 11 fois.
Guarde : 2161, Les minutes [36, 37] apparaissent 13 fois.
Guarde : 613, Les minutes [39] apparaissent 10 fois.
Guarde : 1619, Les minutes [33] apparaissent 18 fois.
Guarde : 883, Les minutes [43, 44, 45, 49] apparaissent 8 fois.
Guarde : 2423, Les minutes [47, 48] apparaissent 9 fois.
Guarde : 1571, Les minutes [54] apparaissent 12 fois.
Guarde : 2801, Les minutes [45] apparaissent 12 fois.
Guarde : 953, Les minutes [21, 24, 25, 26] apparaissent 11 fois.
Guarde : 449, Les minutes [43] apparaissent 12 fois.
Guarde : 919, Les minutes [50, 51, 43, 46, 47, 49] apparaissent 8 fois.
Guarde : 3299, Les minutes [28, 30, 31, 32, 33, 34, 35] apparaissent 11 fois.
Guarde : 1307, Les minutes [29, 30] apparaissent 7 fois.
Guarde : 1607, Les minutes [49, 50, 51, 52] apparaissent 5 fois.
Guarde : 1091, Les minutes [14, 15] appa